# Fine-tune ReactionT5 on a 300k ORD sample (Model 1, Kaggle GPU)

Same recipe as `01_train_reactant_ord_more_data.ipynb` (150k, proven working: `torchrun --nproc_per_node=2`, `--no-augment --learning-rate 5e-5 --num-train-epochs 3`), scaled to ~300,000 ORD reactions (2x the 150k pool).

**Why scale up again after 150k gave a mixed result (RESULTS.md, variant 7):** a paired McNemar significance test between variant 2 (60k) and variant 7 (150k) on the same 300-record ORD eval set found none of the top-1/top-5, exact/core differences statistically significant (all p > 0.05, closest p=0.06). The apparent "more data hurt top-1" pattern in the raw percentages is most likely eval-set noise (n=300), not a real regression -- so scaling data further is not repeating a proven-bad move.

**Do NOT copy `02_train_reactant_ord_250k.ipynb` as a base** -- that notebook's training cell launches with plain `!python` (no `torchrun`), missing the DDP fix, so it would silently fall back to the slower/weaker `DataParallel` path. This notebook is based on `01_train_reactant_ord_more_data.ipynb` instead, which is the one that actually completed successfully with real DDP (confirmed via its `train.log`: `train_runtime=1.418e4s`, `train_steps_per_second=0.972`, `"Training finished"` exactly once).

**Before running:** in the notebook Settings panel (right sidebar) turn on **Internet** and **GPU accelerator** (T4x2). Kaggle's free GPU quota is **30 hours/week**.

**Data:** the 300k-reaction sample was built locally (`build_train_data_ord.py --pool-count 300000`, same seed/eval-exclusion logic as the 60k/150k/250k pools -- independently verified 0 product_smiles overlap with `data/v2_ord_eval_targets.json`, both train and val) and must be uploaded as a **Kaggle Dataset** (`reactants_train.jsonl`, `reactants_val.jsonl`), then added to this notebook as an input (`+ Add Input`, right sidebar).

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
print("Device count:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(f"  Device {i}:", torch.cuda.get_device_name(i))
if torch.cuda.device_count() < 2:
    print("WARNING: fewer than 2 GPUs visible -- the torchrun --nproc_per_node=2 launch below expects 2.")

In [ ]:
import os

if not os.path.isdir("retro-planner"):
    !git clone https://github.com/oleh-kuzmenko/retro-planner.git
%cd retro-planner

In [ ]:
%pip install -q -e ".[local-models,indexing]"

**Input data.** Adjust the dataset slug below to match whatever you named the Kaggle Dataset you uploaded (visible under `/kaggle/input/` once added as an input).

In [ ]:
import os, glob

dataset_slug = "retro-planner-ord-300k"  # @param {type:"string"}

# Kaggle's actual mount path has changed before (plain /kaggle/input/<slug>/ vs
# /kaggle/input/datasets/<owner>/<slug>/) -- confirmed by a real failure on this exact
# dataset (kernel version 1: assertion on the naive path). Search instead of guessing.
candidates = [
    f"/kaggle/input/{dataset_slug}",
    f"/kaggle/input/datasets/kuzmenkooleh/{dataset_slug}",
]
base = next((c for c in candidates if os.path.exists(os.path.join(c, "reactants_train.jsonl"))), None)
if base is None:
    found = glob.glob("/kaggle/input/**/reactants_train.jsonl", recursive=True)
    listing = os.listdir("/kaggle/input") if os.path.isdir("/kaggle/input") else "/kaggle/input MISSING"
    assert found, f"reactants_train.jsonl not found under /kaggle/input -- did you add the dataset as an input? /kaggle/input contents: {listing}"
    base = os.path.dirname(found[0])

train_file = os.path.join(base, "reactants_train.jsonl")
val_file = os.path.join(base, "reactants_val.jsonl")
assert os.path.exists(train_file), f"Not found: {train_file}"
assert os.path.exists(val_file), f"Not found: {val_file}"
print("Resolved base:", base)
print("Train file:", train_file, "--", sum(1 for _ in open(train_file)), "rows")
print("Val file:", val_file, "--", sum(1 for _ in open(val_file)), "rows")

**Cross-session resume on Kaggle.** There's no Drive-style live mount here -- `/kaggle/working` only persists once you **Save Version** ("commit") the notebook, which turns its contents into this notebook's own Output, downloadable as a dataset. To continue training in a later session:

1. This session: train, then **Save Version** before your quota/time runs out. The committed `/kaggle/working/<output_dir_name>` becomes an Output you can download or directly reuse.
2. Next session: either (a) add *this same notebook's* previous Output version as an input (Kaggle lets you pick a specific version's output), or (b) download the `final`/`checkpoint-N` folder and re-upload it as its own small Dataset -- same idea as the Colab notebook's cross-account resume.
3. Point `resume_from_checkpoint_path` below at wherever that folder landed under `/kaggle/input/...`.

Leave `resume_from_checkpoint_path` blank for a first run.

In [ ]:
resume_from_checkpoint_path = ""  # @param {type:"string"}
# e.g. /kaggle/input/model1-ord300k-checkpoint/checkpoint-NNNN  (full Trainer checkpoint -- exact resume)
# or   /kaggle/input/model1-ord300k-checkpoint/final           (weights only -- fresh optimizer/step count)

In [ ]:
output_dir = "/kaggle/working/model1_reactant_ord300k_v2cfg"  # @param {type:"string"}
time_budget_minutes = 600  # @param {type:"number"}
# 3 epochs (matching v2's proven config) over 297k/32 effective-batch is ~27,844 steps. Measured
# (not estimated) DDP rate from the completed 150k run's own train.log: train_steps_per_second=
# 0.972 -- so 27,844 steps / 0.972 steps/s = ~28,647s = ~477min (~7.96h). 600 min (10h) leaves
# ~26% headroom within the 15h free-quota budget for this experiment.

In [ ]:
import os

os.makedirs(output_dir, exist_ok=True)
log_path = f"{output_dir}/train.log"
resume_flag = ["--resume-from-checkpoint", resume_from_checkpoint_path] if resume_from_checkpoint_path else []

!torchrun --nproc_per_node=2 scripts/train_reactant_model_ord.py \
    --train-file "{train_file}" \
    --val-file "{val_file}" \
    --output-dir "{output_dir}" \
    --local-work-dir /kaggle/temp/local_model1_work \
    --no-augment \
    --learning-rate 5e-5 \
    --num-train-epochs 3 \
    --time-budget-minutes {time_budget_minutes} \
    {' '.join(resume_flag)} \
    > "{log_path}" 2>&1
print(f"Done (or paused at time budget). Log: {log_path}")

Same log-redirect reasoning as the Colab notebook: printing per-step output directly in the cell can make the tab unresponsive over a multi-hour run. Since this runs as a Commit job, you don't need to watch it at all -- check back later via `kaggle kernels status <user>/<slug>` (CLI) or the Output tab.

`--local-work-dir` points at `/kaggle/temp` (fast local scratch disk, wiped between sessions) so Trainer's own checkpoint rotation never touches `/kaggle/working` directly; the training script's own `DriveSyncCallback`-style logic still copies out one `latest_checkpoint` folder under `output_dir` after every save. Only rank 0 (of the 2 `torchrun` processes) does this Drive-style sync and the final save (fixed via `RANK`-based rank detection -- see `scripts/train_reactant_model_ord.py`), so there's no risk of the two GPU processes racing to write the same files.

**When done:** download via CLI (`kaggle kernels output <user>/<slug> -p <dest>`) or the Output tab. `output_dir/final` has the model. Evaluate it exactly like the other checkpoints:

```
python scripts/models/run_reactiont5_topk.py \
    --input data/v2_ord_eval_targets.json \
    --t5-model <downloaded_final_dir> \
    --num-beams 10 --output experiments/v2_model1_topk/ord300k_v2cfg_topk.json
```